In [ ]:
import warnings
warnings.filterwarnings( 'ignore' )
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, TimeSeriesSplit
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.impute import SimpleImputer

In [ ]:
partition = 478

In [ ]:
trainpath = f'../../../../../data/top30groups/DNDF_noGeographic/train{partition}.csv'
testpath = f'../../../../../data/top30groups/DNDF_noGeographic/test{partition}.csv'

traindata = pd.read_csv(trainpath, encoding='ISO-8859-1')
testdata = pd.read_csv(testpath, encoding='ISO-8859-1')

if 'attack_date' in traindata.columns:
    traindata = traindata.drop(columns=['attack_date'])

if 'attack_date' in testdata.columns:
    testdata = testdata.drop(columns=['attack_date'])

    print(f'shape train data: ', traindata.shape)
    print(f'shape test data: ', testdata.shape)

In [ ]:
def split_data(dftrain, dftest):
    Ytrain = dftrain['gname']
    Xtrain = dftrain.drop(columns=['gname'])
    Ytest = dftest['gname']
    Xtest = dftest.drop(columns=['gname'])
    return Xtrain, Ytrain, Xtest, Ytest

def find_best_3layer_mlp(Xtrain, Ytrain):
    param_dist = {
        # Each tuple has 3 values: 3 hidden layers
        'hidden_layer_sizes': [
            (50, 50, 50), (100, 50, 25), (100, 100, 50), 
            (150, 100, 50), (200, 150, 100), (300, 200, 100)
        ],
        'activation': ['relu', 'tanh'],
        'solver': ['adam'],
        'alpha': [1e-5, 1e-4, 1e-3, 1e-2],
        'learning_rate_init': [0.0001, 0.001, 0.01],
        'early_stopping': [True],
        'max_iter': [200, 300, 500]  # optional
    }

    mlp = MLPClassifier(random_state=42)

    random_search = RandomizedSearchCV(
        estimator=mlp,
        param_distributions=param_dist,
        n_iter=20,  # Number of sampled configs
        cv=5,  # 5-fold CV
        scoring='accuracy',
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    random_search.fit(Xtrain, Ytrain)
    return random_search.best_estimator_

In [ ]:
from sklearn.base import clone

Xtrain, Ytrain, Xtest, Ytest = split_data(traindata, testdata)
best_mlp = find_best_mlp(Xtrain, Ytrain)
final_mlp = clone(best_mlp)
final_mlp.fit(Xtrain, Ytrain)
y_pred_mlp = final_mlp.predict(Xtest)
accuracy_mlp = accuracy_score(Ytest, y_pred_mlp)
print(f"Accuracy: {accuracy_mlp * 100:.2f}%")

In [ ]:
file_path = os.path.join("results", f"gtd{partition}.txt")

# Make sure the directory exists
os.makedirs("results", exist_ok=True)

# Write a string to the file
with open(file_path, "w") as file:
    file.write(accuracy)

In [ ]:
print(best_mlp)